<b>Business Objective:<b>

For every customer, rank the brands they have not yet purchased based on their likelihood of adopting them in the next campaign window.

This is not a homepage recommendation problem.

This is a cross-brand expansion / multi-brand targeting problem.

In [2]:
import pandas as pd
import os 

In [56]:
path = os.path.abspath(os.path.join(os.getcwd(), os.pardir, "data\\cross-brand expansion\\realistic_cross_brand_ranking_dataset.xlsx")) 
path

'C:\\git\\ML-hands-on-notebooks\\data\\cross-brand expansion\\realistic_cross_brand_ranking_dataset.xlsx'

In [61]:
df = pd.read_excel(path, header=0)
df.head()

,customer_id,feature_cutoff_date,candidate_brand,age,gender,income,city_tier,rfm_score,frequency_6m,avg_spend,...,categories_bought,brands_purchased_before_cutoff,num_brands_purchased,brand_price_segment,brand_style,brand_target_age,brand_popularity,season,campaign_type,label_future_purchase
0,100001,2025-06-30,Brand_B,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Premium,Formal,30-45,0.84,Monsoon,Email,0
1,100001,2025-06-30,Brand_C,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Mid,Sports,20-35,0.66,Monsoon,Email,1
2,100001,2025-06-30,Brand_D,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Premium,Ethnic,25-40,0.81,Monsoon,Email,0
3,100001,2025-06-30,Brand_E,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Budget,Sports,18-30,0.58,Monsoon,Email,0
4,100001,2025-06-30,Brand_F,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Mid,Casual,25-35,0.63,Monsoon,Email,0


In [63]:
df.shape

(39894, 23)

In [78]:
df.columns

Index(['customer_id', 'feature_cutoff_date', 'candidate_brand', 'age',
       'gender', 'income', 'city_tier', 'rfm_score', 'frequency_6m',
       'avg_spend', 'discount_pct', 'return_rate', 'recency_days',
       'categories_bought', 'brands_purchased_before_cutoff',
       'num_brands_purchased', 'brand_price_segment', 'brand_style',
       'brand_target_age', 'brand_popularity', 'season', 'campaign_type',
       'label_future_purchase'],
      dtype='object')

In [64]:
df.describe()

,customer_id,age,income,city_tier,rfm_score,frequency_6m,avg_spend,discount_pct,return_rate,recency_days,num_brands_purchased,brand_popularity,label_future_purchase
count,39894.000000,39894.000000,39894.000000,39894.000000,39894.000000,39894.000000,39894.000000,39894.000000,39894.000000,39894.000000,39894.000000,39894.000000,39894.000000
mean,102501.762947,38.989297,137509.127312,1.994285,0.599875,10.448940,7796.899358,25.078533,0.124724,91.035519,1.937685,0.699858,0.150800
std,1443.984372,12.367513,65296.561262,0.819826,0.230693,5.736704,4095.572748,14.511949,0.072352,51.530666,0.813663,0.092424,0.357858
min,100001.000000,18.000000,25000.000000,1.000000,0.200000,1.000000,805.000000,0.000000,0.000000,1.000000,1.000000,0.550000,0.000000
25%,101250.000000,28.000000,81540.000000,1.000000,0.400000,6.000000,4250.000000,13.000000,0.060000,48.000000,1.000000,0.630000,0.000000
50%,102504.000000,39.000000,137550.000000,2.000000,0.600000,10.000000,7766.000000,25.000000,0.120000,92.000000,2.000000,0.680000,0.000000
75%,103752.000000,50.000000,194441.000000,3.000000,0.800000,15.000000,11334.000000,38.000000,0.190000,136.000000,3.000000,0.790000,0.000000
max,105000.000000,60.000000,249921.000000,3.000000,1.000000,20.000000,15000.000000,50.000000,0.250000,180.000000,3.000000,0.840000,1.000000


In [66]:
df.describe(include=['O'])

,feature_cutoff_date,candidate_brand,gender,categories_bought,brands_purchased_before_cutoff,brand_price_segment,brand_style,brand_target_age,season,campaign_type
count,39894,39894,39894,39894,39894,39894,39894,39894,39894,39894
unique,1,10,2,80,764,4,5,8,4,4
top,2025-06-30,Brand_H,M,"Formal,Ethnic",Brand_D,Mid,Casual,20-35,Monsoon,Push
freq,39894,4075,20182,1125,1728,12019,11905,7969,10473,10172


In [70]:
df["brands_purchased_before_cutoff"].tail(10)

39884    Brand_F,Brand_I,Brand_C
39885    Brand_F,Brand_I,Brand_C
39886    Brand_F,Brand_I,Brand_C
39887    Brand_G,Brand_D,Brand_I
39888    Brand_G,Brand_D,Brand_I
39889    Brand_G,Brand_D,Brand_I
39890    Brand_G,Brand_D,Brand_I
39891    Brand_G,Brand_D,Brand_I
39892    Brand_G,Brand_D,Brand_I
39893    Brand_G,Brand_D,Brand_I
Name: brands_purchased_before_cutoff, dtype: object

In [73]:
df["label_future_purchase"].value_counts(dropna=False)

label_future_purchase
0    33878
1     6016
Name: count, dtype: int64

In [77]:
df["campaign_type"].value_counts(dropna=False)

campaign_type
Push        10172
SMS         10108
Email       10048
WhatsApp     9566
Name: count, dtype: int64

In [75]:
df.groupby("customer_id")["candidate_brand"].count()

customer_id
100001    9
100002    9
100003    7
100004    7
100005    7
         ..
104996    9
104997    8
104998    8
104999    7
105000    7
Name: candidate_brand, Length: 5000, dtype: int64

In [76]:
df[df["customer_id"] == 100001]

,customer_id,feature_cutoff_date,candidate_brand,age,gender,income,city_tier,rfm_score,frequency_6m,avg_spend,...,categories_bought,brands_purchased_before_cutoff,num_brands_purchased,brand_price_segment,brand_style,brand_target_age,brand_popularity,season,campaign_type,label_future_purchase
0,100001,2025-06-30,Brand_B,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Premium,Formal,30-45,0.84,Monsoon,Email,0
1,100001,2025-06-30,Brand_C,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Mid,Sports,20-35,0.66,Monsoon,Email,1
2,100001,2025-06-30,Brand_D,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Premium,Ethnic,25-40,0.81,Monsoon,Email,0
3,100001,2025-06-30,Brand_E,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Budget,Sports,18-30,0.58,Monsoon,Email,0
4,100001,2025-06-30,Brand_F,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Mid,Casual,25-35,0.63,Monsoon,Email,0
5,100001,2025-06-30,Brand_G,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Luxury,Luxury,30-50,0.79,Monsoon,Email,0
6,100001,2025-06-30,Brand_H,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Mid,Ethnic,25-45,0.68,Monsoon,Email,0
7,100001,2025-06-30,Brand_I,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Budget,Formal,25-40,0.55,Monsoon,Email,0
8,100001,2025-06-30,Brand_J,38,M,128500,3,0.24,18,2342,...,"Formal,Luxury",Brand_A,1,Premium,Casual,20-35,0.74,Monsoon,Email,0


# Transaction table

In [115]:
transaction_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir, "data\\cross-brand expansion\\fashion_transactions_3years.xlsx")) 
transactions = pd.read_excel(transaction_path, header=0)
print(transactions.shape)
transactions.head()

(164896, 7)


,customer_id,transaction_date,brand,category,sales_amount,discount_pct,returned
0,100001,2023-07-29,Brand_E,Luxury,1924,37,0
1,100001,2024-04-21,Brand_A,Luxury,10363,1,1
2,100001,2024-03-27,Brand_C,Formal,10154,17,0
3,100001,2025-05-15,Brand_A,Ethnic,5052,9,0
4,100001,2023-07-29,Brand_E,Casual,6724,6,0


In [126]:
transactions["transaction_date"] = pd.to_datetime(transactions["transaction_date"])
print(f" Min transaction date = {transactions["transaction_date"].min()}")
print(f" Max transaction date = {transactions["transaction_date"].max()}")


 Min transaction date = 2023-01-01 00:00:00
 Max transaction date = 2025-12-30 00:00:00


In [156]:
transactions["transaction_year"] = pd.to_datetime(transactions["transaction_date"]).dt.year
transactions[["transaction_date", "transaction_year"]].head()

,transaction_date,transaction_year
0,2023-07-29,2023
1,2024-04-21,2024
2,2024-03-27,2024
3,2025-05-15,2025
4,2023-07-29,2023


In [157]:
transactions["transaction_year"].value_counts()

transaction_year
2024    55123
2023    54894
2025    54879
Name: count, dtype: int64

In [158]:
transactions["customer_id"].nunique()

10000

In [159]:
transactions.isnull().sum()

customer_id         0
transaction_date    0
brand               0
category            0
sales_amount        0
discount_pct        0
returned            0
transaction_year    0
dtype: int64

In [168]:
# Fixed cutoff
cutoffs = [pd.to_datetime("2023-06-01"),
           pd.to_datetime("2023-12-01"),
           pd.to_datetime("2024-06-01"),
           pd.to_datetime("2024-12-01"),
           pd.to_datetime("2025-06-01")]

for cutoff in cutoffs:
    feature_start = cutoff - pd.DateOffset(months=6)
    feature_end = cutoff
    
    label_start = cutoff + pd.DateOffset(months=1)
    label_end = cutoff + pd.DateOffset(months=3)
    print(f"Feature = {feature_start} - {feature_end}; Label = {label_start} - {label_end}")


Feature = 2022-12-01 00:00:00 - 2023-06-01 00:00:00; Label = 2023-07-01 00:00:00 - 2023-09-01 00:00:00
Feature = 2023-06-01 00:00:00 - 2023-12-01 00:00:00; Label = 2024-01-01 00:00:00 - 2024-03-01 00:00:00
Feature = 2023-12-01 00:00:00 - 2024-06-01 00:00:00; Label = 2024-07-01 00:00:00 - 2024-09-01 00:00:00
Feature = 2024-06-01 00:00:00 - 2024-12-01 00:00:00; Label = 2025-01-01 00:00:00 - 2025-03-01 00:00:00
Feature = 2024-12-01 00:00:00 - 2025-06-01 00:00:00; Label = 2025-07-01 00:00:00 - 2025-09-01 00:00:00


In [176]:
cutoffs = pd.date_range("2023-06-30", "2025-09-30", freq="3ME")  # quarterly cutoffs

for cutoff in cutoffs:
    # Feature window: past 6 months
    feature_start = cutoff - pd.DateOffset(months=6)
    feature_end = cutoff
    
    # Label window: next 3 months
    label_start = cutoff + pd.DateOffset(days=1)
    label_end = cutoff + pd.DateOffset(months=3)
    print(f"Feature = {feature_start, feature_end}; Label = {label_start, label_end}")



Feature = (Timestamp('2022-12-30 00:00:00'), Timestamp('2023-06-30 00:00:00')); Label = (Timestamp('2023-07-01 00:00:00'), Timestamp('2023-09-30 00:00:00'))
Feature = (Timestamp('2023-03-30 00:00:00'), Timestamp('2023-09-30 00:00:00')); Label = (Timestamp('2023-10-01 00:00:00'), Timestamp('2023-12-30 00:00:00'))
Feature = (Timestamp('2023-06-30 00:00:00'), Timestamp('2023-12-31 00:00:00')); Label = (Timestamp('2024-01-01 00:00:00'), Timestamp('2024-03-31 00:00:00'))
Feature = (Timestamp('2023-09-30 00:00:00'), Timestamp('2024-03-31 00:00:00')); Label = (Timestamp('2024-04-01 00:00:00'), Timestamp('2024-06-30 00:00:00'))
Feature = (Timestamp('2023-12-30 00:00:00'), Timestamp('2024-06-30 00:00:00')); Label = (Timestamp('2024-07-01 00:00:00'), Timestamp('2024-09-30 00:00:00'))
Feature = (Timestamp('2024-03-30 00:00:00'), Timestamp('2024-09-30 00:00:00')); Label = (Timestamp('2024-10-01 00:00:00'), Timestamp('2024-12-30 00:00:00'))
Feature = (Timestamp('2024-06-30 00:00:00'), Timestamp('20

10 snapshots created from the above feature/label window split.

Of those use 2023 and 2024 for train

use 2025 for test

# Demographics 

In [92]:
demo_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir, "data\\cross-brand expansion\\customer_master.xlsx")) 
demo_df = pd.read_excel(demo_path, header=0)
print(demo_df.shape)
demo_df.head()

(10000, 6)


,customer_id,age,gender,income,city_tier,loyalty_tier
0,100001,37,F,160034,1,Platinum
1,100002,34,F,188319,1,Platinum
2,100003,41,M,80124,2,Platinum
3,100004,55,F,117707,3,Platinum
4,100005,51,F,58606,3,Gold


In [93]:
demo_df["customer_id"].nunique()

10000

In [94]:
demo_df.isnull().sum()

customer_id     0
age             0
gender          0
income          0
city_tier       0
loyalty_tier    0
dtype: int64

# Merge customer features and transactions tables

In [178]:
def create_customer_features(transactions, customers, cutoff_date, feature_window_months=6):
    # Feature window: past 6 months
    feature_start = cutoff_date - pd.DateOffset(months=feature_window_months)
    print(f"Feature window = {feature_start, cutoff_date}")

    history = transactions[(transactions["transaction_date"] > feature_start) &
                           (transactions["transaction_date"] <= cutoff_date)].copy()

    features = (
        history
        .groupby("customer_id")
        .agg(
            total_orders=("transaction_date","count"),
            total_sales=("sales_amount","sum"),
            avg_spend=("sales_amount","mean"),
            avg_discount=("discount_pct","mean"),
            return_rate=("returned","mean"),
            last_purchase=("transaction_date","max"),
            categories=(
                "category",
                lambda x: ",".join(sorted(x.unique()))
            ),
            brands=(
                "brand",
                lambda x: ",".join(sorted(x.unique()))
            ),
            num_brands=(
                "brand",
                "nunique"
            )
        )
        .reset_index()
    )

    features["recency_days"] = (cutoff_date - features["last_purchase"]).dt.days

    features["feature_cutoff_date"] = cutoff_date

    features = customers.merge(
        features,
        on="customer_id",
        how="left"
    )

    return features

In [180]:
cutoffs = pd.date_range("2023-06-30", "2025-09-30", freq="3ME")  # quarterly cutoffs
cutoffs

DatetimeIndex(['2023-06-30', '2023-09-30', '2023-12-31', '2024-03-31',
               '2024-06-30', '2024-09-30', '2024-12-31', '2025-03-31',
               '2025-06-30', '2025-09-30'],
              dtype='datetime64[ns]', freq='3ME')

In [183]:
all_snapshots = []
for cutoff in cutoffs:
    snapshot = create_customer_features(transactions, demo_df, cutoff)
    all_snapshots.append(snapshot)



Feature window = (Timestamp('2022-12-30 00:00:00'), Timestamp('2023-06-30 00:00:00'))
Feature window = (Timestamp('2023-03-30 00:00:00'), Timestamp('2023-09-30 00:00:00'))
Feature window = (Timestamp('2023-06-30 00:00:00'), Timestamp('2023-12-31 00:00:00'))
Feature window = (Timestamp('2023-09-30 00:00:00'), Timestamp('2024-03-31 00:00:00'))
Feature window = (Timestamp('2023-12-30 00:00:00'), Timestamp('2024-06-30 00:00:00'))
Feature window = (Timestamp('2024-03-30 00:00:00'), Timestamp('2024-09-30 00:00:00'))
Feature window = (Timestamp('2024-06-30 00:00:00'), Timestamp('2024-12-31 00:00:00'))
Feature window = (Timestamp('2024-09-30 00:00:00'), Timestamp('2025-03-31 00:00:00'))
Feature window = (Timestamp('2024-12-30 00:00:00'), Timestamp('2025-06-30 00:00:00'))
Feature window = (Timestamp('2025-03-30 00:00:00'), Timestamp('2025-09-30 00:00:00'))


In [184]:
customer_features = pd.concat(
    all_snapshots,
    ignore_index=True
)


In [186]:
# one row = one customer per snapshot approach. 
# quartrly window for 3 years created 10 snapshot per customer. so 10 snapshots * 10,000 customer = 100,000

customer_features.shape

(100000, 17)

In [189]:
customer_features[customer_features["customer_id"] == 100001]

,customer_id,age,gender,income,city_tier,loyalty_tier,total_orders,total_sales,avg_spend,avg_discount,return_rate,last_purchase,categories,brands,num_brands,recency_days,feature_cutoff_date
0,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.750000,15.250000,0.250000,2023-06-13,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30
10000,100001,37,F,160034,1,Platinum,5.0,22730.0,4546.000000,19.400000,0.200000,2023-07-29,"Casual,Luxury,Sports","Brand_E,Brand_F",2.0,63.0,2023-09-30
20000,100001,37,F,160034,1,Platinum,3.0,14968.0,4989.333333,18.666667,0.000000,2023-11-30,"Casual,Ethnic,Luxury",Brand_E,1.0,31.0,2023-12-31
30000,100001,37,F,160034,1,Platinum,2.0,16474.0,8237.000000,15.000000,0.000000,2024-03-27,"Ethnic,Formal","Brand_C,Brand_E",2.0,4.0,2024-03-31
40000,100001,37,F,160034,1,Platinum,3.0,28036.0,9345.333333,18.333333,0.333333,2024-06-22,"Formal,Luxury","Brand_A,Brand_C,Brand_G",3.0,8.0,2024-06-30
50000,100001,37,F,160034,1,Platinum,2.0,17882.0,8941.000000,19.000000,0.500000,2024-06-22,Luxury,"Brand_A,Brand_G",2.0,100.0,2024-09-30
60000,100001,37,F,160034,1,Platinum,1.0,11238.0,11238.000000,31.000000,0.000000,2024-10-06,Sports,Brand_A,1.0,86.0,2024-12-31
70000,100001,37,F,160034,1,Platinum,2.0,14888.0,7444.000000,38.000000,0.000000,2025-01-10,"Luxury,Sports","Brand_A,Brand_G",2.0,80.0,2025-03-31
80000,100001,37,F,160034,1,Platinum,2.0,8702.0,4351.000000,27.000000,0.000000,2025-05-15,"Ethnic,Luxury","Brand_A,Brand_G",2.0,46.0,2025-06-30
90000,100001,37,F,160034,1,Platinum,3.0,14813.0,4937.666667,19.000000,0.333333,2025-08-04,"Ethnic,Formal,Sports","Brand_A,Brand_F",2.0,57.0,2025-09-30


In [190]:
customer_features[customer_features["customer_id"] == 100001]["feature_cutoff_date"].value_counts()

feature_cutoff_date
2023-06-30    1
2023-09-30    1
2023-12-31    1
2024-03-31    1
2024-06-30    1
2024-09-30    1
2024-12-31    1
2025-03-31    1
2025-06-30    1
2025-09-30    1
Name: count, dtype: int64

# Cross Join with Brand Table

In [96]:
brand_path = os.path.abspath(os.path.join(os.getcwd(), os.pardir, "data\\cross-brand expansion\\brand_master.xlsx")) 
brand_df = pd.read_excel(brand_path, header=0)
print(brand_df.shape)
brand_df.head()

(10, 5)


,brand,price_segment,style,target_age,popularity
0,Brand_A,Budget,Casual,18-25,0.72
1,Brand_B,Premium,Formal,30-45,0.84
2,Brand_C,Mid,Sports,20-35,0.66
3,Brand_D,Premium,Ethnic,25-40,0.81
4,Brand_E,Budget,Sports,18-30,0.58


In [191]:
customer_brand = customer_features.merge(
    brand_df,
    how="cross"
)


customer_brand.shape

(1000000, 22)

In [221]:
customer_brand[(customer_brand["customer_id"] == 100001) & (customer_brand["feature_cutoff_date"] == '2023-06-30')]

,customer_id,age,gender,income,city_tier,loyalty_tier,total_orders,total_sales,avg_spend,avg_discount,...,categories,brands,num_brands,recency_days,feature_cutoff_date,brand,price_segment,style,target_age,popularity
0,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_A,Budget,Casual,18-25,0.72
1,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_B,Premium,Formal,30-45,0.84
2,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_C,Mid,Sports,20-35,0.66
3,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_D,Premium,Ethnic,25-40,0.81
4,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_E,Budget,Sports,18-30,0.58
5,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_F,Mid,Casual,25-35,0.63
6,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_G,Luxury,Luxury,30-50,0.79
7,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_H,Mid,Ethnic,25-45,0.68
8,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_I,Budget,Formal,25-40,0.55
9,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Formal,Luxury,Sports","Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_J,Premium,Casual,20-35,0.74


# Remove already purchased brand

In [246]:
all_snapshots = []
for cutoff in cutoffs:
    feature_start = cutoff - pd.DateOffset(months=6)
    print(cutoff)
    owned_pairs = (
        transactions[(transactions["transaction_date"]  > feature_start) &
                     (transactions["transaction_date"]  <= cutoff)
        ][["customer_id", "brand"]]
        .drop_duplicates()
    )
    owned_pairs["feature_cutoff_date"] =cutoff
    all_snapshots.append(owned_pairs)

2023-06-30 00:00:00
2023-09-30 00:00:00
2023-12-31 00:00:00
2024-03-31 00:00:00
2024-06-30 00:00:00
2024-09-30 00:00:00
2024-12-31 00:00:00
2025-03-31 00:00:00
2025-06-30 00:00:00
2025-09-30 00:00:00


In [250]:
owned_brands = pd.concat(
    all_snapshots,
    ignore_index=True
)

owned_brands.shape

(187154, 3)

In [253]:
owned_brands[owned_brands["customer_id"] == 100001]

,customer_id,brand,feature_cutoff_date
0,100001,Brand_E,2023-06-30
1,100001,Brand_F,2023-06-30
18597,100001,Brand_E,2023-09-30
18598,100001,Brand_F,2023-09-30
37411,100001,Brand_E,2023-12-31
56097,100001,Brand_C,2024-03-31
56098,100001,Brand_E,2024-03-31
74749,100001,Brand_A,2024-06-30
74750,100001,Brand_C,2024-06-30
74751,100001,Brand_G,2024-06-30


In [257]:
merged = pd.merge(customer_brand, owned_brands, on=['customer_id', 'brand', 'feature_cutoff_date'], how='left', indicator=True)
merged.shape

(1000000, 23)

In [258]:
anti_join_df = merged[merged['_merge'] == 'left_only'].drop(columns=['_merge'])
anti_join_df.shape

(812846, 22)

In [266]:
anti_join_df[anti_join_df["customer_id"] == 100001].groupby("feature_cutoff_date")["brand"].unique().to_dict()

{Timestamp('2023-06-30 00:00:00'): array(['Brand_A', 'Brand_B', 'Brand_C', 'Brand_D', 'Brand_G', 'Brand_H',
        'Brand_I', 'Brand_J'], dtype=object),
 Timestamp('2023-09-30 00:00:00'): array(['Brand_A', 'Brand_B', 'Brand_C', 'Brand_D', 'Brand_G', 'Brand_H',
        'Brand_I', 'Brand_J'], dtype=object),
 Timestamp('2023-12-31 00:00:00'): array(['Brand_A', 'Brand_B', 'Brand_C', 'Brand_D', 'Brand_F', 'Brand_G',
        'Brand_H', 'Brand_I', 'Brand_J'], dtype=object),
 Timestamp('2024-03-31 00:00:00'): array(['Brand_A', 'Brand_B', 'Brand_D', 'Brand_F', 'Brand_G', 'Brand_H',
        'Brand_I', 'Brand_J'], dtype=object),
 Timestamp('2024-06-30 00:00:00'): array(['Brand_B', 'Brand_D', 'Brand_E', 'Brand_F', 'Brand_H', 'Brand_I',
        'Brand_J'], dtype=object),
 Timestamp('2024-09-30 00:00:00'): array(['Brand_B', 'Brand_C', 'Brand_D', 'Brand_E', 'Brand_F', 'Brand_H',
        'Brand_I', 'Brand_J'], dtype=object),
 Timestamp('2024-12-31 00:00:00'): array(['Brand_B', 'Brand_C', 'Brand_D', '

# Create Labels

In [ ]:
label_window = 90

In [287]:
all_snapshots = []
for cutoff in cutoffs:
    label_start = cutoff + pd.DateOffset(days=1)
    label_end = cutoff + pd.DateOffset(months=3)
    print(label_start, label_end)
    label_pairs = (
        transactions[(transactions["transaction_date"]  > label_start) &
                     (transactions["transaction_date"]  <= label_end)
        ][["customer_id", "brand"]]
        .drop_duplicates()
    )
    label_pairs["label"] = 1
    label_pairs["label_cutoff_date"] =label_end
    label_pairs["feature_cutoff_date"] =cutoff
    all_snapshots.append(label_pairs)

2023-07-01 00:00:00 2023-09-30 00:00:00
2023-10-01 00:00:00 2023-12-30 00:00:00
2024-01-01 00:00:00 2024-03-31 00:00:00
2024-04-01 00:00:00 2024-06-30 00:00:00
2024-07-01 00:00:00 2024-09-30 00:00:00
2024-10-01 00:00:00 2024-12-30 00:00:00
2025-01-01 00:00:00 2025-03-31 00:00:00
2025-04-01 00:00:00 2025-06-30 00:00:00
2025-07-01 00:00:00 2025-09-30 00:00:00
2025-10-01 00:00:00 2025-12-30 00:00:00


In [288]:
label_brands = pd.concat(
    all_snapshots,
    ignore_index=True
)

label_brands.shape

(110517, 5)

In [289]:
label_brands[label_brands["customer_id"] == 100001]

,customer_id,brand,label,label_cutoff_date,feature_cutoff_date
0,100001,Brand_E,1,2023-09-30,2023-06-30
11126,100001,Brand_E,1,2023-12-30,2023-09-30
22068,100001,Brand_C,1,2024-03-31,2023-12-31
33064,100001,Brand_A,1,2024-06-30,2024-03-31
33065,100001,Brand_G,1,2024-06-30,2024-03-31
55236,100001,Brand_A,1,2024-12-30,2024-09-30
66248,100001,Brand_G,1,2025-03-31,2024-12-31
77225,100001,Brand_A,1,2025-06-30,2025-03-31
88234,100001,Brand_A,1,2025-09-30,2025-06-30
88235,100001,Brand_F,1,2025-09-30,2025-06-30


In [290]:
merged = pd.merge(anti_join_df, label_brands, on=['customer_id', 'brand', 'feature_cutoff_date'], how='left', indicator=True)
merged.shape

(812846, 25)

In [310]:
merged[merged['_merge'] == 'left_only']["label"].value_counts(dropna=False)

label
NaN    768040
Name: count, dtype: int64

In [319]:
merged.loc[merged['_merge'] == 'left_only', "label"] = merged.loc[merged['_merge'] == 'left_only', "label"].fillna(0)

In [329]:
merged["label"] = merged["label"].astype(int)
merged["label"].value_counts(dropna=False)

label
0    768040
1     44806
Name: count, dtype: int64

In [330]:
merged["label"].value_counts(dropna=False, normalize=True)

label
0    0.944878
1    0.055122
Name: proportion, dtype: float64

In [331]:
merged[(merged["customer_id"] == 100006) & (merged["feature_cutoff_date"] == '2023-06-30')][["brand", "label"]]

,brand,label
38,Brand_A,0
39,Brand_B,0
40,Brand_C,0
41,Brand_D,0
42,Brand_F,1
43,Brand_G,0
44,Brand_I,0
45,Brand_J,0


In [332]:
# testing - future candidate purchase
future = transactions[(transactions.customer_id == 100006)
            #& (transactions.brand == "Brand_A")
            & (transactions.transaction_date > '2023-06-30')
            & (transactions.transaction_date <= '2023-09-30')
        ]

future["brand"].unique()

array(['Brand_H', 'Brand_F', 'Brand_E'], dtype=object)

In [ ]:
merged.drop(columns=["label_cutoff_date","_merge"], axis=1, inplace=True)

In [333]:
merged.head()

,customer_id,age,gender,income,city_tier,loyalty_tier,total_orders,total_sales,avg_spend,avg_discount,...,brands,num_brands,recency_days,feature_cutoff_date,brand,price_segment,style,target_age,popularity,label
0,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_A,Budget,Casual,18-25,0.72,0
1,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_B,Premium,Formal,30-45,0.84,0
2,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_C,Mid,Sports,20-35,0.66,0
3,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_D,Premium,Ethnic,25-40,0.81,0
4,100001,37,F,160034,1,Platinum,4.0,23367.0,5841.75,15.25,...,"Brand_E,Brand_F",2.0,17.0,2023-06-30,Brand_G,Luxury,Luxury,30-50,0.79,0
